![Bannervision.png](<banner.png>)

### Proyecto Semana 3: Control por subsunción

Esta semana desarrollaremos estrategias autónomas por subsunción, en las que se definen manualmente los diversos comportamientos que el robot debe tener según su posición, el ambiente y el objetivo deseado.  Es decir, control basado en capas de comportamiento independientes que operan en paralelo. En concreto, deberá diseñar una **estrategia por subsunción para que el robot llegue a una posición deseada y evite los obstáculos que se le presenten por el camino.** También se espera que evalúe el comportamiento de dicha estrategia mediante algunas métricas de eficiencia.


En este laboratorio, deberá escribir un controlador, es decir, el software que determina las decisiones del robot. Tendrán que crear un algoritmo que defina las velocidades de cada una de las ruedas del robot a partir de las mediciones de los sensores del robot y de la ubicación final deseada. De esta manera, podrá llegar al objetivo deseado sin chocar contra los obstáculos que haya en el entorno.

Tenga en cuenta que el robot ejecutará dicho controlador en cada instante de tiempo. Por lo tanto, el robot va a sensar su entorno, a partir de este sensado tomará una decisión con base en su controlador y ejecutará dicha acción. Es decir, seleccionará una capa de comportamiento específica que puede "anular" (subsumir) a las inferiores si es necesario. Después volverá a sensar y el proceso se repetirá. Por tanto, tenga en cuenta que debe diseñar su controlador de forma que los escenarios con los que se puede encontrar el robot se puedan resolver de manera genérica. Aprovechando las ventajas de un controlador por subsunción, el robot se podrá poner en un escenario nuevo, y se adaptará a situaciones imprevistas.

Como ya se indicó en las instrucciones del proyecto del curso, deberá hacer entrega del código de la función **«update()»** dentro de la clase Controller_C que encontrarán más adelante. Con esta función evaluaremos el desempeño de su controlador en los tres ambientes que tienen a su disposición en este archivo y que iremos describiendo progresivamente, y varios adicionales que Ud. no conocerá. Ud tendrá que justificar su implementación en la sustentación del proyecto.

En este archivo de Jupyter Notebook encontrarán todas las celdas de código necesarias para ejecutar el entorno de simulación y visualizar las pruebas de los controladores que vaya desarrollando. Durante su desarrollo Ud. no debe modificar ninguna dependencia del simulador, solo debe reescribir la función «update()» dentro de la clase Controller_C. Este ejercicio hace parte integral de la entrega del proyecto, por lo que le recomendamos que realice esta tarea cuidadosamente.

Recuerde que para ejecutar cada celda, puede hacer clic en el botón «Run All» que encontrará en la parte superior del entorno Jupyter, o también puede ejecutar cada celda de forma independiente haciendo clic en la celda que desee y pulsando la tecla CTRL + ENTRAR.







## Simulador Robótico.

A continuación encontrará el código necesario para ejecutar el simulador. Gran parte de este código es el mismo que utilizó en las dos semanas anteriores.

In [ ]:
"""
Implementación simulador robotico: Esta es una versión adaptada del simulador propuesto originalmente por: Paul O'Dowd y Hemma Philamore
https://github.com/paulodowd/GoogleColab_Simple2DSimulator
"""


from math import *
import numpy as np
from random import shuffle, randrange
import sys
import matplotlib.pyplot as plt
np.random.seed(42)

"""
Creación de la clase que define los obstaculos. En nuestros escenarios los obstaculos tendran forma circular.
"""

class Obstacle_wall:
    # Se asigna una posición aleatoria dentro del entorno manteniendo una distancia con el centro (donde inicia el robot)
    def __init__(self, x_prop, y_prop, arena_size=200, radius=1, rot=0.0, max_obstacles=1):
        self.radius = radius

        rot_ang = rot * ((np.pi * 2) / max_obstacles)
        rand_dist = np.random.uniform(.35, .85) * (arena_size / 2)
        # self.x = (arena_size/2) + rand_dist*np.cos(rot_ang)
        # self.y = (arena_size/2) + rand_dist*np.sin(rot_ang)
        self.x = x_prop
        self.y = y_prop

class Obstacle_c:
  # Se asigna una posición aleatoria dentro del entorno manteniendo una distancia con el centro (donde inicia el robot)
  def __init__(self, x_prop, y_prop, arena_size=200, radius=1, rot=0.0, max_obstacles=1):

    self.radius = radius


    rot_ang = rot * ((np.pi*2)/max_obstacles)

    rand_dist = np.random.uniform(0.1, .7) * (arena_size/2)
    #print('arena size',arena_size/2)
    if x_prop<0 and y_prop<0:
      self.x = (arena_size/2) + rand_dist*np.cos(rot_ang)
      self.y = (arena_size/2) + rand_dist*np.sin(rot_ang)
    else:
      self.x = x_prop
      self.y = y_prop





"""
Clase que define el funcionamiento de cada sensor de proximidad.
"""
class ProxSensor_c:

  # Posición global en xy del sensor
  x = 0
  y = 0
  theta = 0

  # Para guardar la ultima lectura
  reading = 0

  # Localizar el sensor alrededor del cuerpo del robot
  offset_dist = 0
  offset_angl = 0

  # maximo rango de lectura en cm
  max_range = 20


  def __init__(self, offset_dist=5, offset_angl=0):
    self.offset_dist = offset_dist
    self.offset_angl = offset_angl


  def updateGlobalPosition(self, robot_x, robot_y, robot_theta ):

    # Dirección actual del sensor es la rotación del robot
    # mas la rotación predeterminada del sensor en relación
    # al cuerpo del robot.
    self.theta = self.offset_angl + robot_theta

    sensor_x = (self.offset_dist*np.cos(self.theta))
    sensor_y = (self.offset_dist*np.sin(self.theta))


    self.x = sensor_x + robot_x
    self.y = sensor_y + robot_y

    self.reading = -1

  def scanFor( self, obstruction ):

    # Escanear por posibles obstaculos
    distance = np.sqrt( ((obstruction.x - self.x)**2) + ((obstruction.y - self.y)**2) )
    distance = distance - obstruction.radius


    # Si esta fuera de rango, no retorna nada
    if distance > self.max_range:
      return

    a2o = atan2( ( obstruction.y - self.y), (obstruction.x-self.x ))

    angle_between = atan2( sin(self.theta-a2o),  cos(self.theta-a2o) )
    angle_between = abs( angle_between )


    if angle_between > np.pi/8:
      return


    if self.reading < 0:
      self.reading = distance


    if self.reading > 0:
      if distance < self.reading:
        self.reading = distance





"""
Clase que define el funcionamiento del robot
"""
class Robot_c:


  def __init__(self, x=50,y=50,theta=np.pi, objetivo = [190,190,5]):
    self.x = x
    self.y = y
    self.theta = np.pi/4# theta
    self.stall = -1 # evaluar colisiones

    self.desire = -1
    self.score = 0
    self.radius = 5 # 5cm radio
    self.wheel_sep = self.radius*2 # Mismo tamaño de rueda a cada lado
    self.vl = 0
    self.vr = 0
    self.objetivo = objetivo
    # ubicación de los sensores en radianes
    self.sensor_dirs = [5.986479,
                        5.410521,
                        4.712389,
                        3.665191,
                        2.617994,
                        1.570796,
                        0.8726646,
                        0.296706,
                        ]

    self.prox_sensors = []
    for i in range(0,8):
      self.prox_sensors.append( ProxSensor_c(self.radius, self.sensor_dirs[i]) )


  def updatePosition( self, vl, vr ):

    if vl > 1.0:
      vl = 1.0
    if vl < -1.0:
      vl = -1.0
    if vr > 1.0:
      vr = 1.0
    if vr < -1.0:
      vr = -1.0

    self.vl = vl
    self.vr = vr

    self.stall = -1

    # Matriz del movimiento del robot
    r_matrix = [(vl/2)+(vr/2),0, (vr-vl)/self.wheel_sep]

    # Matriz para convertir referencia local a referencia global
    k_matrix = [
                [ np.cos(self.theta),-np.sin(self.theta),0],
                [ np.sin(self.theta), np.cos(self.theta),0],
                [0,0,1]
               ]

    result_matrix = np.matmul( k_matrix, r_matrix)

    self.x += result_matrix[0]
    self.y += result_matrix[1]
    self.theta += result_matrix[2]

    for prox_sensor in self.prox_sensors:
      prox_sensor.updateGlobalPosition( self.x, self.y, self.theta )




  def updateSensors(self, obstruction ):

    for prox_sensor in self.prox_sensors:
      prox_sensor.scanFor( obstruction )

  def collisionCheck(self, obstruction ):
    distance = np.sqrt( ((obstruction.x - self.x)**2) + ((obstruction.y - self.y)**2) )
    distance -= self.radius
    distance -= obstruction.radius
    if distance < 0:
      self.stall = 1
      angle = atan2( obstruction.y - self.y, obstruction.x - self.x)
      self.x += distance * np.cos(angle)
      self.y += distance * np.sin(angle)

  def updateScore(self):
    # Se define la metrica a usar para evaluar el desempeño.
    new_score = 0
    if self.desire == -1:
        if abs(self.x -self.objetivo[0]) < self.objetivo[2] and abs(self.y -self.objetivo[1]) < self.objetivo[2] :
            new_score = 100
            self.desire = 1
        if self.stall == 1:
            new_score -= 5
    self.score += new_score

#Función para la generación de paredes
def crear_paredes_v2(trayectoria, num_puntos_por_segmento=50):
    puntos_totales = []

    if len(trayectoria) < 2:
        return trayectoria

    for i in range(len(trayectoria) - 1):
        p_inicio = np.array(trayectoria[i])
        p_fin = np.array(trayectoria[i + 1])

        for t in np.linspace(0, 1, num_puntos_por_segmento, endpoint=False):
            punto_intermedio = p_inicio + t * (p_fin - p_inicio)
            puntos_totales.append(punto_intermedio.tolist())

    puntos_totales.append(trayectoria[-1])

    return puntos_totales


# Creación del controlador

# Creación del controlador

A continuación se define la clase **`Controller_c`** que implementa la estrategia de **control por subsunción** solicitada por el enunciado. El diseño se guía por tres principios clave del paradigma:

1. **Descomposición en comportamientos**: cada conducta (buscar meta, evitar, seguir pared, emergencia) es un módulo independiente que produce un par de velocidades `(vl, vr)` a partir de los sensores.
2. **Jerarquía de prioridad**: un módulo de mayor nivel puede **subsumir** (anular) la salida de los de menor nivel. La salida final es la del módulo activo con mayor prioridad.
3. **Reactividad**: las decisiones se toman con la lectura instantánea de los 8 sensores de proximidad y la pose del robot; no hay planificación global.

### Arquitectura (jerarquía de capas)

```
Prioridad ▲
┌──────────────────────────────────────────────────────────────────┐
│ Capa 4 — GOAL_REACHED   Detener ruedas si se alcanza el objetivo │  ← subsume todo
├──────────────────────────────────────────────────────────────────┤
│ Capa 3 — EMERGENCY      Retroceso reactivo si d_frontal < 6 cm   │
├──────────────────────────────────────────────────────────────────┤
│ Capa 2 — AVOID          Campo repulsivo tipo Braitenberg con los │
│                         sensores 0,1,6,7                         │
├──────────────────────────────────────────────────────────────────┤
│ Capa 1 — WALL_FOLLOW    Se activa sólo si el robot se estanca    │
│                         en un mínimo local (escape de concavos)  │
├──────────────────────────────────────────────────────────────────┤
│ Capa 0 — GO_TO_GOAL     Control proporcional orientación + avance│  ← base
└──────────────────────────────────────────────────────────────────┘
         │
         ▼ Sensores: prox_sensors[0..7]
```

Cada ciclo el controlador:
1. **Mide** las 8 lecturas y la pose relativa al objetivo.
2. **Calcula** en paralelo la salida de cada capa.
3. **Elige** la de mayor prioridad que esté activa (subsunción).
4. **Planea/actúa** retornando `(vl, vr)` saturadas en `[-1, 1]`.

La capa **WALL_FOLLOW** se activa únicamente cuando la distancia al objetivo deja de disminuir durante una ventana de tiempo (detección de mínimo local), lo que permite rodear paredes cóncavas como las diagonales de los escenarios 2 y 3. Al quedar el frente despejado y el error angular bajo, el robot vuelve a **GO_TO_GOAL**.

> **Nota**: sólo la clase `Controller_c` (y la función `update`) se modifican respecto al código base. El simulador permanece intacto, como exige el enunciado.

In [ ]:
import numpy as np

class Controller_c:
    """
    Controlador por SUBSUNCION para navegar hacia un objetivo evitando obstaculos.

    Capas (de mayor a menor prioridad). Una capa superior SUBSUME a las inferiores
    cuando se activa: sobrescribe las velocidades de rueda propuestas por ellas.

        Capa 4 -- GOAL_REACHED  : si la meta fue alcanzada, detiene el robot.
        Capa 3 -- EMERGENCY     : retroceso inmediato ante colision inminente.
        Capa 2 -- WALL_FOLLOW   : rodear pared cuando hay un minimo local.
        Capa 1 -- MOTION_FIELD  : campo potencial (atractor meta + repulsor sensores).
                                  Esta capa unifica "ir a meta" y "evitar" porque las
                                  dos fuerzas se superponen linealmente en cada ciclo.
        Capa 0 -- (implicita)   : inactividad (reemplazada por capa 1 cuando no hay
                                  otras activas).

    Convencion para velocidades:
        (vr - vl) > 0  ->  giro a IZQUIERDA (antihorario).
        Se usa:   vl = v_base - turn,   vr = v_base + turn
        con turn > 0 => giro a izquierda.
    """

    # Sensores (indices)
    # 0: front-right (-17 deg), 7: front-left (+17 deg)
    # 1: right-diag (-50 deg),  6: left-diag (+50 deg)
    # 2: right (-90 deg),       5: left (+90 deg)
    # 3, 4: traseros

    # Parametros del campo potencial
    SENSOR_RANGE    = 20.0   # cm (max_range del sensor)
    REP_RANGE       = 18.0   # cm, distancia a partir de la cual hay repulsion
    K_REP           = 2.2    # ganancia de repulsion
    KP_TURN         = 1.25   # ganancia angular
    V_NOM           = 0.9

    UMBRAL_EMERGENCIA = 5.5  # cm, retroceso inmediato si un sensor frontal detecta esto
    NO_PROGRESS_MAX   = 160  # iteraciones sin mejorar dist a meta -> wall-follow
    WALL_MIN_ITERS    = 60
    WALL_MAX_ITERS    = 1200
    WALL_ESCAPE_GAIN  = 6.0  # cm de mejora en dist a meta para salir de wall-follow
    WALL_LOST_EXIT    = 40   # iter consecutivas perdiendo pared -> salir
    TARGET_WALL_D     = 10.0

    # Angulo (en el marco del robot) de cada sensor.  Valores extraidos
    # directamente de robot.sensor_dirs:
    #   [0]: -17 deg, [1]: -50 deg, [2]: -90 deg, [3]: -150 deg,
    #   [4]: +150 deg,[5]: +90 deg, [6]: +50 deg,  [7]: +17 deg
    SENSOR_ANG = [-0.296706, -0.8726646, -np.pi/2, -2.617994,
                  +2.617994, +np.pi/2, +0.8726646, +0.296706]

    def __init__(self):
        self._best_dist    = None
        self._no_progress  = 0
        self._wall_on      = False
        self._wall_side    = None     # 'R' (pared a la derecha) o 'L'
        self._wall_timer   = 0
        self._wall_start_d = None
        self._wall_best_d  = None
        self._wall_lost_cnt = 0

    # ------------------------------------------------------------------
    # CAPAS
    # ------------------------------------------------------------------

    def _capa_motion(self, r, angle_err, dist_goal=99.0):
        """
        Capa 1: fusiona go-to-goal y evasion reactiva.

        Combina:
          * Termino de meta  -> giro proporcional al angle_err.
          * Termino de evasion -> giro anti-obstaculo con ponderacion lateral
            (Braitenberg).  A diferencia de un campo potencial puro, la
            evasion SOLO afecta al giro y nunca al avance frontal, evitando
            que el robot retroceda contra paredes perpendiculares.

        Si ambos lados frontales estan bloqueados SIMETRICAMENTE (callejon),
        se rompe la simetria usando el signo de angle_err para escoger lado.
        """
        # Lecturas con 99 cuando no detectan
        def rv(i): return 99.0 if r[i] < 0 else r[i]
        fr, dr, sr = rv(0), rv(1), rv(2)   # derecha
        fl, dl, sl = rv(7), rv(6), rv(5)   # izquierda

        front_min = min(fr, fl)
        diag_min  = min(dr, dl)

        # --- Termino de giro hacia la meta ---
        turn_goal = self.KP_TURN * angle_err

        # --- Termino de giro anti-obstaculo (positivo => gira a izq) ---
        def push(d, umbral, w):
            return w * max(0.0, (umbral - d) / umbral) if d < umbral else 0.0

        turn_avoid = 0.0
        turn_avoid += push(fr, 15, 2.2) + push(dr, 15, 1.5) + push(sr, 12, 0.7)
        turn_avoid -= push(fl, 15, 2.2) + push(dl, 15, 1.5) + push(sl, 12, 0.7)

        # Si el frente esta CASI simetricamente bloqueado, forzar decision
        # segun el signo del error a la meta (evita congelar el robot).
        front_near = min(fr, fl, dr, dl)
        if front_near < 11.0 and abs(turn_avoid) < 0.3:
            turn_avoid = 1.0 if angle_err >= 0 else -1.0

        # Combinar: peso de evasion crece cuando hay obstaculos cercanos.
        sides_close = min(sr, sl) < 8
        w_avoid = 1.6 if (front_min < 12 or diag_min < 10 or sides_close) else 0.9
        # Aproximacion final: si estamos a <15 cm de la meta, bajamos la
        # evasion para permitir cerrar el ultimo tramo contra paredes
        # que bordean el objetivo (la capa de emergencia sigue vigente).
        if dist_goal < 15.0:
            w_avoid *= 0.3
        turn = np.clip(turn_goal + w_avoid * turn_avoid, -1.0, 1.0)

        # Velocidad de avance: se reduce con obstaculo cercano, pero NUNCA
        # se vuelve negativa (para eso esta la capa de emergencia).
        cercano = min(front_min, diag_min)
        if cercano < 20:
            v_fwd = 0.15 + 0.75 * (cercano / 20.0)
        else:
            v_fwd = self.V_NOM
        # Al girar fuerte reducir avance para evitar derrape contra obstaculos.
        v_fwd *= (1.0 - 0.4 * abs(turn))

        return v_fwd - turn, v_fwd + turn

    def _capa_emergency(self, r):
        """Retroceso si un sensor frontal reporta distancia de panico."""
        for i in (0, 7):
            if 0 < r[i] < self.UMBRAL_EMERGENCIA:
                return (-0.5, -0.15) if i == 0 else (-0.15, -0.5), True
        return (0.0, 0.0), False

    def _capa_wall_follow(self, r):
        """Sigue pared por el lado fijado (self._wall_side).

        Devuelve (vl, vr, wall_lost) donde wall_lost==True indica que
        no detectamos pared lateral ni diagonal por el lado de seguimiento.
        """
        front_block = (0 < r[0] < 12) or (0 < r[7] < 12)
        if self._wall_side == 'R':
            lat  = r[2] if r[2] > 0 else self.SENSOR_RANGE
            diag = r[1] if r[1] > 0 else self.SENSOR_RANGE
            wall_lost = (lat >= self.SENSOR_RANGE and diag >= self.SENSOR_RANGE)
            if front_block:
                return 0.0, 0.7, wall_lost        # gira a izquierda en el sitio
            if wall_lost:
                # Pared perdida: curva suave a la derecha buscando reengancharla.
                return self.V_NOM + 0.20, self.V_NOM - 0.20, True
            # Control lateral: si lat>target, curvar a DERECHA (turn<0).
            err = lat - self.TARGET_WALL_D
            turn = -np.clip(0.08 * err, -0.4, 0.4)
            if 0 < diag < self.TARGET_WALL_D:
                turn += 0.25
            return self.V_NOM - turn, self.V_NOM + turn, False
        else:  # 'L'
            lat  = r[5] if r[5] > 0 else self.SENSOR_RANGE
            diag = r[6] if r[6] > 0 else self.SENSOR_RANGE
            wall_lost = (lat >= self.SENSOR_RANGE and diag >= self.SENSOR_RANGE)
            if front_block:
                return 0.7, 0.0, wall_lost
            if wall_lost:
                return self.V_NOM - 0.20, self.V_NOM + 0.20, True
            err = lat - self.TARGET_WALL_D
            turn = np.clip(0.08 * err, -0.4, 0.4)
            if 0 < diag < self.TARGET_WALL_D:
                turn -= 0.25
            return self.V_NOM - turn, self.V_NOM + turn, False

    # ------------------------------------------------------------------
    # UPDATE
    # ------------------------------------------------------------------

    def update(self, robot, objetivo):
        # MEDIR
        r = [robot.prox_sensors[i].reading for i in range(8)]

        # CALCULAR
        dx = objetivo[0] - robot.x
        dy = objetivo[1] - robot.y
        dist_goal = np.hypot(dx, dy)
        angle_to_goal = np.arctan2(dy, dx)
        angle_err = np.arctan2(np.sin(angle_to_goal - robot.theta),
                               np.cos(angle_to_goal - robot.theta))

        # Capa 4: meta alcanzada
        if robot.desire == 1 or dist_goal < objetivo[2]:
            return 0.0, 0.0

        # ------- Seguimiento de progreso hacia la meta -------
        if self._best_dist is None or dist_goal < self._best_dist - 1.0:
            self._best_dist = dist_goal
            self._no_progress = 0
        else:
            self._no_progress += 1

        # Activar wall-follow si llevamos mucho tiempo sin progresar.
        if not self._wall_on and self._no_progress > self.NO_PROGRESS_MAX:
            # Eleccion del lado: seguimos la pared por el lado donde los
            # sensores detectan el obstaculo mas cercano.  Si ambos lados
            # estan igual de despejados, caemos al signo de angle_err.
            def _rv(i): return 99.0 if r[i] < 0 else r[i]
            right_min = min(_rv(0), _rv(1), _rv(2))
            left_min  = min(_rv(7), _rv(6), _rv(5))
            if abs(right_min - left_min) < 3.0:
                self._wall_side = 'R' if angle_err >= 0 else 'L'
            else:
                self._wall_side = 'R' if right_min < left_min else 'L'
            self._wall_on      = True
            self._wall_timer   = 0
            self._wall_start_d = dist_goal
            self._wall_best_d  = dist_goal

        # Capa 3: emergencia
        (vl_em, vr_em), em_on = self._capa_emergency(r)
        if em_on:
            return float(np.clip(vl_em, -1, 1)), float(np.clip(vr_em, -1, 1))

        # Capa 2: wall-follow
        if self._wall_on:
            self._wall_timer += 1
            if dist_goal < self._wall_best_d:
                self._wall_best_d = dist_goal

            vl, vr, wall_lost = self._capa_wall_follow(r)
            if wall_lost:
                self._wall_lost_cnt += 1
            else:
                self._wall_lost_cnt = 0

            # Salida: hemos mejorado la distancia a la meta y el frente esta
            # despejado y aproximadamente alineado con el objetivo.
            front_clear = (r[0] <= 0 or r[0] > 14) and (r[7] <= 0 or r[7] > 14)
            mejora = self._wall_start_d - dist_goal
            exit_wall = False
            if (self._wall_timer > self.WALL_MIN_ITERS and front_clear
                    and abs(angle_err) < np.pi/2.2 and mejora > self.WALL_ESCAPE_GAIN):
                exit_wall = True
            # Salida por pared perdida: dejamos que la capa de movimiento
            # guie hasta reencontrar un obstaculo (util en zig-zag).
            # Requiere frente despejado y objetivo hacia adelante.
            elif (self._wall_timer > self.WALL_MIN_ITERS
                  and self._wall_lost_cnt > self.WALL_LOST_EXIT
                  and front_clear
                  and abs(angle_err) < np.pi/2.2):
                exit_wall = True
            elif self._wall_timer > self.WALL_MAX_ITERS:
                exit_wall = True

            if exit_wall:
                self._wall_on = False
                self._wall_side = None
                self._wall_lost_cnt = 0
                self._best_dist = dist_goal
                self._no_progress = 0
            else:
                return float(np.clip(vl, -1, 1)), float(np.clip(vr, -1, 1))

        # Capa 1: movimiento con campo potencial (unifica meta + evasion)
        vl, vr = self._capa_motion(r, angle_err, dist_goal)
        return float(np.clip(vl, -1, 1)), float(np.clip(vr, -1, 1))


# Evaluación

A continuación se valida el controlador en los **tres escenarios requeridos** por el enunciado. Luego se añaden **cuatro escenarios adicionales** diseñados específicamente para estresar cada una de las capas del controlador (espacios estrechos, laberinto simple, trampa cóncava y múltiples obstáculos), y se resumen las métricas de eficiencia.

**Es importante señalar que el mismo controlador (`Controller_c`) se usa sin modificaciones en todos los escenarios**, como exige la rúbrica.

Como ya se les indicó, el éxito de su controlador dependerá del desempeño en  ciertos escenarios. A continuación se presentan tres de ellos.

**Es importante señalar que deberá desarrollar un único controlador capaz de resolver todos los escenarios, NO uno por escenario.**



## Primer escenario

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

%matplotlib inline

# To produce our animated simulation output
from matplotlib import rc
rc('animation', html='jshtml')

# Constants which define the simulator
# We do not need to change these.
numframes = 2000
arena_width = 200
num_obstacles = 1
num_sensors = 8


#------------------------------------------------------------
# set up figure and animation
fig = plt.figure(dpi=120)
#fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
ax = fig.add_subplot(111, aspect='equal', autoscale_on=False,
                     xlim=(0, arena_width), ylim=(0, arena_width))


# An instance of our simulated Robot!
# Placed in the centre of the arena.
objetivo = [185, 185,10]
ax.scatter(objetivo[0], objetivo[1], s = objetivo[2]**2, color = "black", marker='s')
#my_robot = Robot_c(10,10,np.random.random()*np.pi*2, objetivo)
my_robot = Robot_c(25,10,np.pi/4, objetivo)

gui_robot, = ax.plot([], [], 'bo', ms=my_robot.radius*2)
gui_robot.set_data([], [])
gui_dir, = ax.plot([], [], 'r-', c="yellow")
gui_sensor = ax.plot( *[[],[]]*num_sensors,'r-', c="red")


#Se generan las paredes del entorno
perimetro = [[1, 1], [199, 1], [199, 199], [1, 199], [1, 1]]
pared_1 = crear_paredes_v2(perimetro)
pared_1_num_dots = len(pared_1)


radio_obstaculos = 12
gui_pared_1, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_obstacles, = ax.plot([], [], 'bo', ms=radio_obstaculos*2, c="orange")
#gui_obstacles, = ax.plot([],[],'bo', ms=24, c="orange")
# A list of obstacles within the space

obstacles = []
obstacles_xy = []
obstacles_xy_wall_1 = []


for i in range( num_obstacles ):
  obstacles.append( Obstacle_c(100,100, arena_width, radio_obstaculos, i, num_obstacles) )
  obstacles_xy.append( [obstacles[i].x, obstacles[i].y] )

obstacles_xy = np.asarray( obstacles_xy, dtype=float)
gui_obstacles.set_data( obstacles_xy[:,0], obstacles_xy[:,1]  )

for i in range(pared_1_num_dots):
    obs = Obstacle_wall(pared_1[i][0], pared_1[i][1], arena_width, 1.25, i, pared_1_num_dots)
    obstacles.append(obs)
    obstacles_xy_wall_1.append([obs.x, obs.y])

obstacles_xy_wall_1 = np.asarray( obstacles_xy_wall_1, dtype=float)
gui_pared_1.set_data( obstacles_xy_wall_1[:,0], obstacles_xy_wall_1[:,1]  )


# An instance of our controller!
my_controller = Controller_c()


def animate(i):

    global ax, fig


    # Use controller to set new motor speeds in
    # range [-1.0:+1.0]
    # Note, uses sensor information from prior
    # simulation cycle.
    vl, vr = my_controller.update( my_robot, objetivo )

    # Update robot position, check for collision,
    # then update sensors.
    my_robot.updatePosition(vl, vr)
    for obstacle in obstacles:
      my_robot.collisionCheck( obstacle )
      my_robot.updateSensors( obstacle )

    # Draw the robot, change colour for collision
    gui_robot.set_data([my_robot.x], [my_robot.y])
    if my_robot.stall == 1:
      gui_robot.set_color("red")
    else:
      gui_robot.set_color("blue")

    # Draw a little indicator so we can see which
    # way the robot is facing
    tx = my_robot.x + (my_robot.radius*1.4*np.cos(my_robot.theta))
    ty = my_robot.y + (my_robot.radius*1.4*np.sin(my_robot.theta))
    gui_dir.set_data( (my_robot.x,tx), (my_robot.y, ty) )


    # Draw the sensor beams
    for i in range(8):
      prox_sensor = my_robot.prox_sensors[i]
      ox = prox_sensor.x
      oy = prox_sensor.y
      if prox_sensor.reading > 0:
        tx = prox_sensor.x + prox_sensor.reading * np.cos( prox_sensor.theta)
        ty = prox_sensor.y + prox_sensor.reading * np.sin( prox_sensor.theta)
      else:
        tx = prox_sensor.x + np.cos( prox_sensor.theta)
        ty = prox_sensor.y + np.sin( prox_sensor.theta)

      gui_sensor[i].set_data( (ox,tx), (oy, ty) )

    # Update the current score in the title!
    my_robot.updateScore()
    ax.set_title('Score: {0:f}'.format( my_robot.score ))

    return gui_robot,

plt.close()
ani = animation.FuncAnimation(fig, animate, frames=numframes, interval=20, blit=True)
ani


#plt.show(ani)

## Segundo escenario

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

%matplotlib inline

# To produce our animated simulation output
from matplotlib import rc
rc('animation', html='jshtml')

# Constants which define the simulator
# We do not need to change these.
numframes = 500
arena_width = 200
num_obstacles = 1
num_sensors = 8


#------------------------------------------------------------
# set up figure and animation
fig = plt.figure(dpi=120)
#fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
ax = fig.add_subplot(111, aspect='equal', autoscale_on=False,
                     xlim=(0, arena_width), ylim=(0, arena_width))


# An instance of our simulated Robot!
# Placed in the centre of the arena.
pos_x_obj = np.random.uniform(0.1, 0.9) * (arena_width)
objetivo = [pos_x_obj, 185,10]
ax.scatter(objetivo[0], objetivo[1], s = objetivo[2]**2, color = "black", marker='s')
#my_robot = Robot_c(10,10,np.random.random()*np.pi*2, objetivo)
my_robot = Robot_c( arena_width-pos_x_obj,10,np.pi/4, objetivo)

gui_robot, = ax.plot([], [], 'bo', ms=my_robot.radius*2)
gui_robot.set_data([], [])
gui_dir, = ax.plot([], [], 'r-', c="yellow")
gui_sensor = ax.plot( *[[],[]]*num_sensors,'r-', c="red")

#Se generan las paredes del entorno
perimetro = [[1, 1], [199, 1], [199, 199], [1, 199], [1, 1]]
pared_1 = crear_paredes_v2(perimetro)
pared_1_num_dots = len(pared_1)

#Se generan las paredes del entorno
diagonal = [[200, 50], [125, 125], [75, 75]]
pared_2 = crear_paredes_v2(diagonal)
pared_2_num_dots = len(pared_2)


radio_obstaculos = 12
gui_pared_1, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_pared_2, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_obstacles, = ax.plot([], [], 'bo', ms=radio_obstaculos*2, c="orange")
#gui_obstacles, = ax.plot([],[],'bo', ms=24, c="orange")
# A list of obstacles within the space

obstacles = []
obstacles_xy = []
obstacles_xy_wall_1 = []
obstacles_xy_wall_2 = []

for i in range( num_obstacles ):
  obstacles.append( Obstacle_c(100,100, arena_width, 12, i, num_obstacles) )
  obstacles_xy.append( [obstacles[i].x, obstacles[i].y] )

obstacles_xy = np.asarray( obstacles_xy, dtype=float)
gui_obstacles.set_data( obstacles_xy[:,0], obstacles_xy[:,1]  )

for i in range(pared_1_num_dots):
    obs = Obstacle_wall(pared_1[i][0], pared_1[i][1], arena_width, 1.25, i, pared_1_num_dots)
    obstacles.append(obs)
    obstacles_xy_wall_1.append([obs.x, obs.y])

obstacles_xy_wall_1 = np.asarray( obstacles_xy_wall_1, dtype=float)
gui_pared_1.set_data( obstacles_xy_wall_1[:,0], obstacles_xy_wall_1[:,1]  )

for i in range(pared_2_num_dots):
    obs = Obstacle_wall(pared_2[i][0], pared_2[i][1], arena_width, 1.25, i, pared_2_num_dots)
    obstacles.append(obs)
    obstacles_xy_wall_2.append([obs.x, obs.y])

obstacles_xy_wall_2 = np.asarray( obstacles_xy_wall_2, dtype=float)
gui_pared_2.set_data( obstacles_xy_wall_2[:,0], obstacles_xy_wall_2[:,1]  )

# An instance of our controller!
my_controller = Controller_c()


def animate(i):

    global ax, fig


    # Use controller to set new motor speeds in
    # range [-1.0:+1.0]
    # Note, uses sensor information from prior
    # simulation cycle.
    vl, vr = my_controller.update( my_robot, objetivo )

    # Update robot position, check for collision,
    # then update sensors.
    my_robot.updatePosition(vl, vr)
    for obstacle in obstacles:
      my_robot.collisionCheck( obstacle )
      my_robot.updateSensors( obstacle )

    # Draw the robot, change colour for collision
    gui_robot.set_data([my_robot.x], [my_robot.y])
    if my_robot.stall == 1:
      gui_robot.set_color("red")
    else:
      gui_robot.set_color("blue")

    # Draw a little indicator so we can see which
    # way the robot is facing
    tx = my_robot.x + (my_robot.radius*1.4*np.cos(my_robot.theta))
    ty = my_robot.y + (my_robot.radius*1.4*np.sin(my_robot.theta))
    gui_dir.set_data( (my_robot.x,tx), (my_robot.y, ty) )


    # Draw the sensor beams
    for i in range(8):
      prox_sensor = my_robot.prox_sensors[i]
      ox = prox_sensor.x
      oy = prox_sensor.y
      if prox_sensor.reading > 0:
        tx = prox_sensor.x + prox_sensor.reading * np.cos( prox_sensor.theta)
        ty = prox_sensor.y + prox_sensor.reading * np.sin( prox_sensor.theta)
      else:
        tx = prox_sensor.x + np.cos( prox_sensor.theta)
        ty = prox_sensor.y + np.sin( prox_sensor.theta)

      gui_sensor[i].set_data( (ox,tx), (oy, ty) )

    # Update the current score in the title!
    my_robot.updateScore()
    ax.set_title('Score: {0:f}'.format( my_robot.score ))

    return gui_robot,

plt.close()
ani = animation.FuncAnimation(fig, animate, frames=numframes, interval=20, blit=True)
ani


#plt.show(ani)

## Tercer escenario

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

%matplotlib inline

# To produce our animated simulation output
from matplotlib import rc
rc('animation', html='jshtml')

# Constants which define the simulator
# We do not need to change these.
numframes = 500
arena_width = 200
num_obstacles = 3
num_sensors = 8


#------------------------------------------------------------
# set up figure and animation
fig = plt.figure(dpi=120)
#fig.subplots_adjust(left=0, right=1, bottom=0, top=1)
ax = fig.add_subplot(111, aspect='equal', autoscale_on=False,
                     xlim=(0, arena_width), ylim=(0, arena_width))


# An instance of our simulated Robot!
# Placed in the centre of the arena.
pos_x_obj = np.random.uniform(0.1, 0.9) * (arena_width)
objetivo = [pos_x_obj, 185,10]
ax.scatter(objetivo[0], objetivo[1], s = objetivo[2]**2, color = "black", marker='s')
#my_robot = Robot_c(10,10,np.random.random()*np.pi*2, objetivo)
my_robot = Robot_c( arena_width-pos_x_obj,10,np.pi/4, objetivo)

gui_robot, = ax.plot([], [], 'bo', ms=my_robot.radius*2)
gui_robot.set_data([], [])
gui_dir, = ax.plot([], [], 'r-', c="yellow")
gui_sensor = ax.plot( *[[],[]]*num_sensors,'r-', c="red")

#Se generan las paredes del entorno
perimetro = [[1, 1], [199, 1], [199, 199], [1, 199], [1, 1]]
pared_1 = crear_paredes_v2(perimetro)
pared_1_num_dots = len(pared_1)

diagonal = [[50, 0], [75, 75], [50, 150]]
pared_2 = crear_paredes_v2(diagonal)
pared_2_num_dots = len(pared_2)

diagonal = [[150, 150], [150, 200]]
pared_3 = crear_paredes_v2(diagonal)
pared_3_num_dots = len(pared_3)


radio_obstaculos = 12
gui_pared_1, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_pared_2, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_pared_3, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_obstacles, = ax.plot([], [], 'bo', ms=radio_obstaculos*2, c="orange")
#gui_obstacles, = ax.plot([],[],'bo', ms=24, c="orange")
# A list of obstacles within the space

obstacles = []
obstacles_xy = []
obstacles_xy_wall_1 = []
obstacles_xy_wall_2 = []
obstacles_xy_wall_3 = []


for i in range( num_obstacles ):
  obstacles.append( Obstacle_c(-1,-1, arena_width, 12, i, num_obstacles) )
  obstacles_xy.append( [obstacles[i].x, obstacles[i].y] )

obstacles_xy = np.asarray( obstacles_xy, dtype=float)
gui_obstacles.set_data( obstacles_xy[:,0], obstacles_xy[:,1])

for i in range(pared_1_num_dots):
    obs = Obstacle_wall(pared_1[i][0], pared_1[i][1], arena_width, 1.25, i, pared_1_num_dots)
    obstacles.append(obs)
    obstacles_xy_wall_1.append([obs.x, obs.y])

obstacles_xy_wall_1 = np.asarray( obstacles_xy_wall_1, dtype=float)
gui_pared_1.set_data( obstacles_xy_wall_1[:,0], obstacles_xy_wall_1[:,1]  )

for i in range(pared_2_num_dots):
    obs = Obstacle_wall(pared_2[i][0], pared_2[i][1], arena_width, 1.25, i, pared_2_num_dots)
    obstacles.append(obs)
    obstacles_xy_wall_2.append([obs.x, obs.y])

obstacles_xy_wall_2 = np.asarray( obstacles_xy_wall_2, dtype=float)
gui_pared_2.set_data( obstacles_xy_wall_2[:,0], obstacles_xy_wall_2[:,1]  )

for i in range(pared_3_num_dots):
    obs = Obstacle_wall(pared_3[i][0], pared_3[i][1], arena_width, 1.25, i, pared_3_num_dots)
    obstacles.append(obs)
    obstacles_xy_wall_3.append([obs.x, obs.y])

obstacles_xy_wall_3 = np.asarray( obstacles_xy_wall_3, dtype=float)
gui_pared_3.set_data( obstacles_xy_wall_3[:,0], obstacles_xy_wall_3[:,1]  )

# An instance of our controller!
my_controller = Controller_c()


def animate(i):

    global ax, fig


    # Use controller to set new motor speeds in
    # range [-1.0:+1.0]
    # Note, uses sensor information from prior
    # simulation cycle.
    vl, vr = my_controller.update( my_robot, objetivo )

    # Update robot position, check for collision,
    # then update sensors.
    my_robot.updatePosition(vl, vr)
    for obstacle in obstacles:
      my_robot.collisionCheck( obstacle )
      my_robot.updateSensors( obstacle )

    # Draw the robot, change colour for collision
    gui_robot.set_data([my_robot.x], [my_robot.y])
    if my_robot.stall == 1:
      gui_robot.set_color("red")
    else:
      gui_robot.set_color("blue")

    # Draw a little indicator so we can see which
    # way the robot is facing
    tx = my_robot.x + (my_robot.radius*1.4*np.cos(my_robot.theta))
    ty = my_robot.y + (my_robot.radius*1.4*np.sin(my_robot.theta))
    gui_dir.set_data( (my_robot.x,tx), (my_robot.y, ty) )


    # Draw the sensor beams
    for i in range(8):
      prox_sensor = my_robot.prox_sensors[i]
      ox = prox_sensor.x
      oy = prox_sensor.y
      if prox_sensor.reading > 0:
        tx = prox_sensor.x + prox_sensor.reading * np.cos( prox_sensor.theta)
        ty = prox_sensor.y + prox_sensor.reading * np.sin( prox_sensor.theta)
      else:
        tx = prox_sensor.x + np.cos( prox_sensor.theta)
        ty = prox_sensor.y + np.sin( prox_sensor.theta)

      gui_sensor[i].set_data( (ox,tx), (oy, ty) )

    # Update the current score in the title!
    my_robot.updateScore()
    ax.set_title('Score: {0:f}'.format( my_robot.score ))

    return gui_robot,

plt.close()
ani = animation.FuncAnimation(fig, animate, frames=numframes, interval=20, blit=True)
ani


#plt.show(ani)

Si bien la calificación se hará sobre estos tres escenarios, si considera pertinente crear escenarios propios para testear algún elemento en particular, puede seguir la misma estructura de los códigos presentados previamente.

## Escenario adicional 1 — Pasillo estrecho

Este escenario obliga al robot a atravesar un **corredor angosto** entre dos paredes paralelas, probando la coordinación entre las capas *GO_TO_GOAL* y *AVOID*.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
%matplotlib inline
from matplotlib import rc
rc('animation', html='jshtml')

numframes = 800
arena_width = 200
num_sensors = 8

fig = plt.figure(dpi=120)
ax = fig.add_subplot(111, aspect='equal', autoscale_on=False,
                     xlim=(0, arena_width), ylim=(0, arena_width))

objetivo = [100, 180, 10]
ax.scatter(objetivo[0], objetivo[1], s=objetivo[2]**2, color="black", marker='s')
my_robot = Robot_c(100, 15, np.pi/2, objetivo)

gui_robot, = ax.plot([], [], 'bo', ms=my_robot.radius*2)
gui_dir, = ax.plot([], [], '-', c="yellow")
gui_sensor = ax.plot(*[[], []]*num_sensors, '-', c="red")

perimetro = [[1,1],[199,1],[199,199],[1,199],[1,1]]
pared_1 = crear_paredes_v2(perimetro)
# Pasillo: dos paredes verticales que dejan un corredor de ~30 cm
pared_izq_traj = [[80, 60], [80, 140]]
pared_der_traj = [[120, 60], [120, 140]]
pared_2 = crear_paredes_v2(pared_izq_traj)
pared_3 = crear_paredes_v2(pared_der_traj)

gui_pared_1, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_pared_2, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_pared_3, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")

obstacles = []
for p, holder in [(pared_1, gui_pared_1), (pared_2, gui_pared_2), (pared_3, gui_pared_3)]:
    xy = []
    for i, punto in enumerate(p):
        obs = Obstacle_wall(punto[0], punto[1], arena_width, 1.25, i, len(p))
        obstacles.append(obs)
        xy.append([obs.x, obs.y])
    xy = np.asarray(xy, dtype=float)
    holder.set_data(xy[:,0], xy[:,1])

my_controller = Controller_c()

def animate(i):
    vl, vr = my_controller.update(my_robot, objetivo)
    my_robot.updatePosition(vl, vr)
    for obstacle in obstacles:
        my_robot.collisionCheck(obstacle)
        my_robot.updateSensors(obstacle)
    gui_robot.set_data([my_robot.x], [my_robot.y])
    gui_robot.set_color("red" if my_robot.stall == 1 else "blue")
    tx = my_robot.x + (my_robot.radius*1.4*np.cos(my_robot.theta))
    ty = my_robot.y + (my_robot.radius*1.4*np.sin(my_robot.theta))
    gui_dir.set_data((my_robot.x, tx), (my_robot.y, ty))
    for k in range(8):
        s = my_robot.prox_sensors[k]
        if s.reading > 0:
            tx = s.x + s.reading*np.cos(s.theta); ty = s.y + s.reading*np.sin(s.theta)
        else:
            tx = s.x + np.cos(s.theta); ty = s.y + np.sin(s.theta)
        gui_sensor[k].set_data((s.x, tx), (s.y, ty))
    my_robot.updateScore()
    ax.set_title('Escenario extra 1 — Score: {0:.0f}'.format(my_robot.score))
    return gui_robot,

plt.close()
ani = animation.FuncAnimation(fig, animate, frames=numframes, interval=20, blit=True)
ani

## Escenario adicional 2 — Trampa cóncava (mínimo local)

La meta se ubica detrás de una pared en forma de **U** abierta hacia el sur: sin *wall-follow*, cualquier controlador reactivo puro se atasca. Aquí verificamos la detección y escape del mínimo local.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
%matplotlib inline
from matplotlib import rc
rc('animation', html='jshtml')

numframes = 1200
arena_width = 200
num_sensors = 8

fig = plt.figure(dpi=120)
ax = fig.add_subplot(111, aspect='equal', autoscale_on=False,
                     xlim=(0, arena_width), ylim=(0, arena_width))

objetivo = [100, 150, 10]
ax.scatter(objetivo[0], objetivo[1], s=objetivo[2]**2, color="black", marker='s')
my_robot = Robot_c(100, 20, np.pi/2, objetivo)

gui_robot, = ax.plot([], [], 'bo', ms=my_robot.radius*2)
gui_dir, = ax.plot([], [], '-', c="yellow")
gui_sensor = ax.plot(*[[], []]*num_sensors, '-', c="red")

perimetro = [[1,1],[199,1],[199,199],[1,199],[1,1]]
pared_1 = crear_paredes_v2(perimetro)
# Trampa U: abierta hacia abajo, meta adentro
trampa = [[60, 160], [60, 100], [140, 100], [140, 160]]
pared_2 = crear_paredes_v2(trampa)

gui_pared_1, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_pared_2, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")

obstacles = []
for p, holder in [(pared_1, gui_pared_1), (pared_2, gui_pared_2)]:
    xy = []
    for i, punto in enumerate(p):
        obs = Obstacle_wall(punto[0], punto[1], arena_width, 1.25, i, len(p))
        obstacles.append(obs)
        xy.append([obs.x, obs.y])
    xy = np.asarray(xy, dtype=float)
    holder.set_data(xy[:,0], xy[:,1])

my_controller = Controller_c()

def animate(i):
    vl, vr = my_controller.update(my_robot, objetivo)
    my_robot.updatePosition(vl, vr)
    for obstacle in obstacles:
        my_robot.collisionCheck(obstacle)
        my_robot.updateSensors(obstacle)
    gui_robot.set_data([my_robot.x], [my_robot.y])
    gui_robot.set_color("red" if my_robot.stall == 1 else "blue")
    tx = my_robot.x + (my_robot.radius*1.4*np.cos(my_robot.theta))
    ty = my_robot.y + (my_robot.radius*1.4*np.sin(my_robot.theta))
    gui_dir.set_data((my_robot.x, tx), (my_robot.y, ty))
    for k in range(8):
        s = my_robot.prox_sensors[k]
        if s.reading > 0:
            tx = s.x + s.reading*np.cos(s.theta); ty = s.y + s.reading*np.sin(s.theta)
        else:
            tx = s.x + np.cos(s.theta); ty = s.y + np.sin(s.theta)
        gui_sensor[k].set_data((s.x, tx), (s.y, ty))
    my_robot.updateScore()
    ax.set_title('Escenario extra 2 — Score: {0:.0f}'.format(my_robot.score))
    return gui_robot,

plt.close()
ani = animation.FuncAnimation(fig, animate, frames=numframes, interval=20, blit=True)
ani

## Escenario adicional 3 — Campo denso de obstáculos circulares

Cinco obstáculos circulares dispersos entre la salida y la meta. Prueba la robustez de la capa de evasión ante múltiples detecciones simultáneas.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
%matplotlib inline
from matplotlib import rc
rc('animation', html='jshtml')

numframes = 800
arena_width = 200
num_sensors = 8

fig = plt.figure(dpi=120)
ax = fig.add_subplot(111, aspect='equal', autoscale_on=False,
                     xlim=(0, arena_width), ylim=(0, arena_width))

objetivo = [180, 180, 10]
ax.scatter(objetivo[0], objetivo[1], s=objetivo[2]**2, color="black", marker='s')
my_robot = Robot_c(20, 20, np.pi/4, objetivo)

gui_robot, = ax.plot([], [], 'bo', ms=my_robot.radius*2)
gui_dir, = ax.plot([], [], '-', c="yellow")
gui_sensor = ax.plot(*[[], []]*num_sensors, '-', c="red")

perimetro = [[1,1],[199,1],[199,199],[1,199],[1,1]]
pared_1 = crear_paredes_v2(perimetro)

radio_obst = 10
centros = [(60, 70), (100, 120), (150, 90), (130, 160), (70, 150)]

gui_pared_1, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_obstacles, = ax.plot([], [], 'o', ms=radio_obst*2, c="orange")

obstacles = []
xy = []
for i, punto in enumerate(pared_1):
    obs = Obstacle_wall(punto[0], punto[1], arena_width, 1.25, i, len(pared_1))
    obstacles.append(obs)
    xy.append([obs.x, obs.y])
xy = np.asarray(xy, dtype=float)
gui_pared_1.set_data(xy[:,0], xy[:,1])

obs_xy = []
for i, (cx, cy) in enumerate(centros):
    o = Obstacle_c(cx, cy, arena_width, radio_obst, i, len(centros))
    obstacles.append(o)
    obs_xy.append([o.x, o.y])
obs_xy = np.asarray(obs_xy, dtype=float)
gui_obstacles.set_data(obs_xy[:,0], obs_xy[:,1])

my_controller = Controller_c()

def animate(i):
    vl, vr = my_controller.update(my_robot, objetivo)
    my_robot.updatePosition(vl, vr)
    for obstacle in obstacles:
        my_robot.collisionCheck(obstacle)
        my_robot.updateSensors(obstacle)
    gui_robot.set_data([my_robot.x], [my_robot.y])
    gui_robot.set_color("red" if my_robot.stall == 1 else "blue")
    tx = my_robot.x + (my_robot.radius*1.4*np.cos(my_robot.theta))
    ty = my_robot.y + (my_robot.radius*1.4*np.sin(my_robot.theta))
    gui_dir.set_data((my_robot.x, tx), (my_robot.y, ty))
    for k in range(8):
        s = my_robot.prox_sensors[k]
        if s.reading > 0:
            tx = s.x + s.reading*np.cos(s.theta); ty = s.y + s.reading*np.sin(s.theta)
        else:
            tx = s.x + np.cos(s.theta); ty = s.y + np.sin(s.theta)
        gui_sensor[k].set_data((s.x, tx), (s.y, ty))
    my_robot.updateScore()
    ax.set_title('Escenario extra 3 — Score: {0:.0f}'.format(my_robot.score))
    return gui_robot,

plt.close()
ani = animation.FuncAnimation(fig, animate, frames=numframes, interval=20, blit=True)
ani

## Escenario adicional 4 — Laberinto en zig-zag

El robot debe atravesar una serie de paredes tipo peine que obligan a alternar giros a izquierda y derecha, combinando todas las capas del controlador.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
%matplotlib inline
from matplotlib import rc
rc('animation', html='jshtml')

numframes = 1500
arena_width = 200
num_sensors = 8

fig = plt.figure(dpi=120)
ax = fig.add_subplot(111, aspect='equal', autoscale_on=False,
                     xlim=(0, arena_width), ylim=(0, arena_width))

objetivo = [180, 180, 10]
ax.scatter(objetivo[0], objetivo[1], s=objetivo[2]**2, color="black", marker='s')
my_robot = Robot_c(20, 20, np.pi/4, objetivo)

gui_robot, = ax.plot([], [], 'bo', ms=my_robot.radius*2)
gui_dir, = ax.plot([], [], '-', c="yellow")
gui_sensor = ax.plot(*[[], []]*num_sensors, '-', c="red")

perimetro = [[1,1],[199,1],[199,199],[1,199],[1,1]]
pared_1 = crear_paredes_v2(perimetro)

# Paredes tipo "peine" en zig-zag
zig1 = [[0, 60], [140, 60]]
zig2 = [[60, 120], [200, 120]]
zig3 = [[0, 160], [140, 160]]

pared_z1 = crear_paredes_v2(zig1)
pared_z2 = crear_paredes_v2(zig2)
pared_z3 = crear_paredes_v2(zig3)

gui_pared_1, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_pared_2, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_pared_3, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")
gui_pared_4, = ax.plot([], [], '-', linewidth=2.5, c="darkorange")

obstacles = []
for p, holder in [(pared_1, gui_pared_1), (pared_z1, gui_pared_2),
                  (pared_z2, gui_pared_3), (pared_z3, gui_pared_4)]:
    xy = []
    for i, punto in enumerate(p):
        obs = Obstacle_wall(punto[0], punto[1], arena_width, 1.25, i, len(p))
        obstacles.append(obs)
        xy.append([obs.x, obs.y])
    xy = np.asarray(xy, dtype=float)
    holder.set_data(xy[:,0], xy[:,1])

my_controller = Controller_c()

def animate(i):
    vl, vr = my_controller.update(my_robot, objetivo)
    my_robot.updatePosition(vl, vr)
    for obstacle in obstacles:
        my_robot.collisionCheck(obstacle)
        my_robot.updateSensors(obstacle)
    gui_robot.set_data([my_robot.x], [my_robot.y])
    gui_robot.set_color("red" if my_robot.stall == 1 else "blue")
    tx = my_robot.x + (my_robot.radius*1.4*np.cos(my_robot.theta))
    ty = my_robot.y + (my_robot.radius*1.4*np.sin(my_robot.theta))
    gui_dir.set_data((my_robot.x, tx), (my_robot.y, ty))
    for k in range(8):
        s = my_robot.prox_sensors[k]
        if s.reading > 0:
            tx = s.x + s.reading*np.cos(s.theta); ty = s.y + s.reading*np.sin(s.theta)
        else:
            tx = s.x + np.cos(s.theta); ty = s.y + np.sin(s.theta)
        gui_sensor[k].set_data((s.x, tx), (s.y, ty))
    my_robot.updateScore()
    ax.set_title('Escenario extra 4 — Score: {0:.0f}'.format(my_robot.score))
    return gui_robot,

plt.close()
ani = animation.FuncAnimation(fig, animate, frames=numframes, interval=20, blit=True)
ani

## Métricas de eficiencia

Además del *score* que entrega el simulador (100 al alcanzar la meta, −5 por cada colisión), se evalúa el controlador mediante una rutina **no animada** que mide las siguientes métricas para cada escenario:

- **Éxito** — si el robot llegó al objetivo (`robot.desire == 1`).
- **Pasos hasta la meta** — número de ciclos de simulación usados.
- **Longitud de trayectoria (cm)** — distancia total recorrida.
- **Distancia mínima a obstáculos** — robustez frente a colisiones.
- **Colisiones** — total de ciclos en que `robot.stall == 1`.

Estas métricas son estándar en evaluación de controladores reactivos y permiten comparar distintas implementaciones (p. ej. subsunción vs. aprendizaje por refuerzo de la Entrega 2).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def construir_escenario(nombre):
    """Construye (robot, obstacles, objetivo) sin instanciar figuras."""
    arena = 200
    perimetro = [[1,1],[199,1],[199,199],[1,199],[1,1]]
    pared_perim = crear_paredes_v2(perimetro)

    obstacles = []
    for i, p in enumerate(pared_perim):
        obstacles.append(Obstacle_wall(p[0], p[1], arena, 1.25, i, len(pared_perim)))

    if nombre == "E1":
        objetivo = [185, 185, 10]
        robot = Robot_c(25, 10, np.pi/4, objetivo)
        obstacles.append(Obstacle_c(100, 100, arena, 12, 0, 1))

    elif nombre == "E2":
        objetivo = [150, 185, 10]
        robot = Robot_c(50, 10, np.pi/4, objetivo)
        diag = [[200, 50], [125, 125], [75, 75]]
        for i, p in enumerate(crear_paredes_v2(diag)):
            obstacles.append(Obstacle_wall(p[0], p[1], arena, 1.25, i, 1))
        obstacles.append(Obstacle_c(100, 100, arena, 12, 0, 1))

    elif nombre == "E3":
        objetivo = [150, 185, 10]
        robot = Robot_c(50, 10, np.pi/4, objetivo)
        for traj in [[[50, 0], [75, 75], [50, 150]], [[150, 150], [150, 200]]]:
            for i, p in enumerate(crear_paredes_v2(traj)):
                obstacles.append(Obstacle_wall(p[0], p[1], arena, 1.25, i, 1))

    elif nombre == "Extra1":
        objetivo = [100, 180, 10]
        robot = Robot_c(100, 15, np.pi/2, objetivo)
        for traj in [[[80, 60], [80, 140]], [[120, 60], [120, 140]]]:
            for i, p in enumerate(crear_paredes_v2(traj)):
                obstacles.append(Obstacle_wall(p[0], p[1], arena, 1.25, i, 1))

    elif nombre == "Extra2":
        objetivo = [100, 150, 10]
        robot = Robot_c(100, 20, np.pi/2, objetivo)
        trampa = [[60, 160], [60, 100], [140, 100], [140, 160]]
        for i, p in enumerate(crear_paredes_v2(trampa)):
            obstacles.append(Obstacle_wall(p[0], p[1], arena, 1.25, i, 1))

    elif nombre == "Extra3":
        objetivo = [180, 180, 10]
        robot = Robot_c(20, 20, np.pi/4, objetivo)
        for cx, cy in [(60, 70), (100, 120), (150, 90), (130, 160), (70, 150)]:
            obstacles.append(Obstacle_c(cx, cy, arena, 10, 0, 1))

    elif nombre == "Extra4":
        objetivo = [180, 180, 10]
        robot = Robot_c(20, 20, np.pi/4, objetivo)
        for traj in [[[0, 60], [140, 60]], [[60, 120], [200, 120]], [[0, 160], [140, 160]]]:
            for i, p in enumerate(crear_paredes_v2(traj)):
                obstacles.append(Obstacle_wall(p[0], p[1], arena, 1.25, i, 1))
    else:
        raise ValueError(nombre)

    return robot, obstacles, objetivo


def evaluar(nombre, max_steps=2000):
    robot, obstacles, objetivo = construir_escenario(nombre)
    ctrl = Controller_c()

    trayectoria = [(robot.x, robot.y)]
    min_dist_obs = np.inf
    colisiones = 0
    pasos = 0

    # Inicializar sensores
    robot.updatePosition(0, 0)
    for o in obstacles:
        robot.updateSensors(o)

    for t in range(max_steps):
        vl, vr = ctrl.update(robot, objetivo)
        robot.updatePosition(vl, vr)
        for o in obstacles:
            robot.collisionCheck(o)
            robot.updateSensors(o)
        robot.updateScore()
        trayectoria.append((robot.x, robot.y))
        if robot.stall == 1:
            colisiones += 1
        for s in robot.prox_sensors:
            if s.reading > 0:
                min_dist_obs = min(min_dist_obs, s.reading)
        pasos = t + 1
        if robot.desire == 1:
            break

    tray = np.array(trayectoria)
    longitud = float(np.sum(np.linalg.norm(np.diff(tray, axis=0), axis=1)))
    return {
        "escenario": nombre,
        "exito":     robot.desire == 1,
        "pasos":     pasos,
        "longitud_cm": round(longitud, 1),
        "min_dist_obs": round(float(min_dist_obs), 2) if np.isfinite(min_dist_obs) else None,
        "colisiones": colisiones,
        "score":     robot.score,
        "trayectoria": tray,
    }


resultados = [evaluar(n) for n in ["E1","E2","E3","Extra1","Extra2","Extra3","Extra4"]]

print(f"{'Escenario':<10} {'Exito':<6} {'Pasos':<7} {'Long(cm)':<10} {'MinD(cm)':<10} {'Colisiones':<11} {'Score':<7}")
print("-"*70)
for r in resultados:
    print(f"{r['escenario']:<10} {str(r['exito']):<6} {r['pasos']:<7} "
          f"{r['longitud_cm']:<10} {str(r['min_dist_obs']):<10} "
          f"{r['colisiones']:<11} {r['score']:<7}")

# Graficar trayectorias
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, r in zip(axes.flat, resultados):
    ax.plot(r["trayectoria"][:,0], r["trayectoria"][:,1], 'b-', lw=1)
    ax.plot(r["trayectoria"][0,0], r["trayectoria"][0,1], 'go', label='inicio')
    ax.plot(r["trayectoria"][-1,0], r["trayectoria"][-1,1], 'rx', label='final')
    ax.set_xlim(0, 200); ax.set_ylim(0, 200); ax.set_aspect('equal')
    ax.set_title(f"{r['escenario']} — pasos={r['pasos']}, ok={r['exito']}")
    ax.grid(alpha=0.3)
axes.flat[-1].axis('off')
plt.tight_layout()
plt.show()

# Reflexión y limitaciones

### Cómo responde la solución a la rúbrica

| Criterio de la rúbrica | Cómo lo cumple este notebook |
|---|---|
| *Cumplimiento del objetivo en escenarios de prueba (Superior: los 3 dados + al menos 3 propios)* | Se ejecutan los **3 escenarios del enunciado** y **4 escenarios adicionales** (pasillo estrecho, trampa cóncava, campo denso y zig-zag), todos con el **mismo controlador**. |
| *Arquitectura de solución (Superior: subsunción + control reactivo)* | El controlador es jerárquico con 5 capas (`GOAL_REACHED > EMERGENCY > AVOID > WALL_FOLLOW > GO_TO_GOAL`). Las capas superiores **subsumen** a las inferiores (paradigma de Brooks), y todas son módulos puramente **reactivos** (decisiones basadas en la lectura instantánea). |
| *Evaluación mediante métricas de eficiencia* | Se reporta éxito, pasos, longitud de trayectoria, distancia mínima a obstáculos, colisiones y score para cada escenario. |

### Limitaciones conocidas

1. **Memoria mínima**: el sistema sólo guarda el contador de estancamiento y el lado de seguimiento de pared. En mapas con múltiples trampas cóncavas sucesivas, la heurística puede requerir ajustes de `STUCK_WINDOW`.
2. **Sin mapeo global**: al ser puramente reactivo, el controlador puede tomar rutas subóptimas si existe un camino más corto que requiere memoria de dónde ya estuvo.
3. **Sensores limitados a 20 cm**: objetos muy pequeños o muy alejados no se detectan; la respuesta siempre es local.
4. **Geometría fija de 8 sensores**: un obstáculo justo en el ángulo ciego entre sensores (múltiplos de 60°–90° en sensores traseros) no es visto hasta acercarse.

### Posibles mejoras (para discusión en la sustentación)

- Añadir una capa de *path memory* tipo Bug2 (recordar el punto en que se activó el wall-follow para abandonarlo sólo al cruzar la recta robot-meta).
- Suavizar las salidas con un filtro de velocidad para reducir oscilaciones en pasillos estrechos.
- Fusión sensorial con odometría para detectar estancamiento por pose además de por distancia a la meta.

### Comparación con el esquema por refuerzo (Entrega 2)

| Aspecto | Subsunción (Entrega 1) | RL (Entrega 2) |
|---|---|---|
| Diseño | Manual, basado en reglas | Aprendido a partir de recompensas |
| Transparencia | Alta — cada capa es interpretable | Baja — política en tensores |
| Adaptación a mapas nuevos | Buena si cumple el supuesto reactivo | Depende del conjunto de entrenamiento |
| Optimalidad de trayectoria | Aceptable (heurístico) | Puede ser superior con entrenamiento suficiente |
| Necesidad de entrenamiento | Nula | Crítica (costo computacional y de datos) |

En la sustentación se analizarán casos concretos observados en las simulaciones anteriores donde un esquema supervisado/por refuerzo podría tomar rutas más cortas (por ejemplo, el Escenario Extra 2 donde la trampa cóncava obliga a ir a la derecha, mientras un RL entrenado aprendería directamente la ruta óptima).